# 17. 업로드한 그림 3개 읽어 보기

[프로젝트 적용] 소나타와 같은 Supabase images 저장소에서 개인 그림을 조회합니다.
[프로젝트 추가] 2026-10-02 승인 후 PDF 44쪽 2개·45쪽 1개를 개인 경로에 업로드하고, 인증 다운로드와 공개 주소의 실제 파일 값을 PDF 원본과 대조했습니다. 총 63,745바이트입니다.
이 노트북은 읽기만 합니다. 업로드·DB 수정·모델 실행은 하지 않습니다. 개인 코드와 로컬 .env가 준비된 프로젝트 환경의 커널을 선택하세요.

In [ ]:
from pathlib import Path
import sys

# 실행 폴더가 notebooks여도 프로젝트 루트를 찾아 같은 개인 코드를 불러옵니다.
project_folder = next((folder for folder in (Path.cwd(), *Path.cwd().parents)
                       if (folder / 'src/car_search_rag/zzong_santafe_lag').is_dir()), None)
if project_folder is None:
    raise FileNotFoundError('프로젝트 폴더에서 이 노트북을 열어 주세요.')
sys.path.insert(0, str(project_folder / 'src'))
from car_search_rag.zzong_santafe_lag.image_storage_service import SampleImageStorageService


In [ ]:
# prepare는 PDF·DB·Storage를 읽고 이미 있는 그림이 원본과 같은지 확인합니다.
# 비밀키는 출력하지 않으며, 파일 업로드나 DB 변경은 없습니다.
service = SampleImageStorageService()
report = service.prepare()
print('준비 확인:', report['preflight_passed'])
print('Storage 그림 파일 합계:', report['total_file_bytes'], '바이트')
for item in report['objects']:
    print('PDF 쪽:', item['pdf_page'], '| 파일:', item['file_name'])
    print('이미 저장돼 있나요?:', item['reusable_existing'])
    print('Storage 경로:', item['storage_path'])


In [ ]:
from IPython.display import Image, display

# 공개 주소로 그림을 보여 줍니다. 크기 조절은 화면 표시용이며 원본을 바꾸지 않습니다.
for plan in service.plans:
    image = plan['image']
    print('PDF 쪽:', image['pdf_page_number'], '| 원래 파일명:', image['file_name'])
    print('DB 업로드 상태:', image['upload_status'])
    if image['upload_status'] != 'uploaded' or not plan['existing']:
        raise ValueError('업로드와 DB 연결 상태부터 확인하세요.')
    url = service.client.storage.from_('images').get_public_url(plan['path'])
    display(Image(url=url, width=700))


## 무엇을 확인했나요?

Storage는 그림 파일, DB images는 파일의 위치, record_images는 주제와 그림 설명을 보관합니다.
PC의 JPEG와 PDF에서 다시 꺼낸 JPEG는 압축 차이가 있을 수 있어, 업로드는 PDF에서 추출한 원본 bytes를 사용했습니다. PC 파일을 수정하거나 업로드한 그림 자체를 임베딩한 것은 아닙니다.
이번 결과는 그림 3개의 업로드·조회·DB 연결 완료입니다. 전체 그림 설명·전체 PDF 저장·생성 답변 완료는 아닙니다.
공식 조회 안내: https://supabase.com/docs/reference/python/storage-from-getpublicurl